In [ ]:
import pygame
import numpy as np
import h5py
import time
import random
from typing import List, Tuple, Optional
from enum import Enum
from PIL import Image

class Direction(Enum):
    UP = (0, -1)
    DOWN = (0, 1)
    LEFT = (-1, 0)
    RIGHT = (1, 0)
    NONE = (0, 0)

class GameState:
    def __init__(self, width=10, height=10):
        self.width = width
        self.height = height
        self.reset()
    
    def reset(self):
        # Create maze (1 = wall, 0 = empty, 2 = pellet, 3 = power pellet)
        self.maze = np.zeros((self.height, self.width), dtype=np.int8)
        self._create_simple_maze()
        
        # Game entities - only red ghost now
        self.pacman_pos = [5, 7]  # [x, y] - Adjusted for 10x10 grid
        self.ghosts = [
            {'pos': [4, 4], 'dir': Direction.UP, 'color': 0, 'stuck_counter': 0},    # Red ghost only
        ]
        
        self.score = 0
        self.pellets_remaining = np.sum(self.maze == 2)
        self.power_mode = 0  # Frames remaining in power mode
        self.game_over = False
        self.won = False
        
    def _create_simple_maze(self):
        # Fill with pellets initially
        self.maze.fill(2)
        
        # Create outer walls
        self.maze[0, :] = 1
        self.maze[-1, :] = 1
        self.maze[:, 0] = 1
        self.maze[:, -1] = 1
        
        # Create a few large, simple obstructions
        # Block in the top left
        self.maze[2:4, 2:4] = 1
        # Block in the bottom right
        self.maze[6:8, 6:8] = 1
        
        # Clear center area for ghost
        self.maze[4:6, 4:6] = 0
        
        # Clear pacman starting area
        self.maze[7, 5] = 0
        
        # Add power pellets in the corners
        self.maze[1, 1] = 3
        self.maze[1, 8] = 3
        self.maze[8, 1] = 3
        self.maze[8, 8] = 3
    
    def is_valid_move(self, pos: List[int], direction: Direction) -> bool:
        new_x = pos[0] + direction.value[0]
        new_y = pos[1] + direction.value[1]
        
        # Wrap around horizontally
        new_x = new_x % self.width
        
        # Check bounds and walls
        if new_y < 0 or new_y >= self.height:
            return False
        if self.maze[new_y, new_x] == 1:
            return False
        return True
    
    def move_pacman(self, direction: Direction):
        if self.game_over:
            return
        
        if self.is_valid_move(self.pacman_pos, direction):
            self.pacman_pos[0] = (self.pacman_pos[0] + direction.value[0]) % self.width
            self.pacman_pos[1] += direction.value[1]
            
            # Check for pellets
            x, y = self.pacman_pos
            if self.maze[y, x] == 2:  # Regular pellet
                self.maze[y, x] = 0
                self.score += 10
                self.pellets_remaining -= 1
            elif self.maze[y, x] == 3:  # Power pellet
                self.maze[y, x] = 0
                self.score += 50
                #self.power_mode = 60  # 60 frames of power mode
                self.pellets_remaining -= 1
            
            # Check win condition
            if self.pellets_remaining == 0:
                self.won = True
                self.game_over = True
    
    def get_ghost_possible_actions(self, ghost_idx: int) -> List[int]:
        """Get valid actions for a ghost (0=stay, 1=up, 2=down, 3=left, 4=right)"""
        if ghost_idx >= len(self.ghosts):
            return []
            
        ghost = self.ghosts[ghost_idx]
        possible_actions = []
        
        directions = [Direction.NONE, Direction.UP, Direction.DOWN, Direction.LEFT, Direction.RIGHT]
        
        for i, direction in enumerate(directions):
            if i == 0:  # NONE - always possible but discouraged
                possible_actions.append(i)
            elif self.is_valid_move(ghost['pos'], direction):
                possible_actions.append(i)
        
        return possible_actions
    
    def apply_ghost_action(self, ghost_idx: int, action: int) -> bool:
        """Apply action to ghost. Returns True if action was valid."""
        if ghost_idx >= len(self.ghosts):
            return False
            
        ghost = self.ghosts[ghost_idx]
        directions = [Direction.NONE, Direction.UP, Direction.DOWN, Direction.LEFT, Direction.RIGHT]
        
        if action < 0 or action >= len(directions):
            return False
            
        direction = directions[action]
        
        if action == 0:  # NONE - stay in place
            return True
        elif self.is_valid_move(ghost['pos'], direction):
            ghost['dir'] = direction
            ghost['pos'][0] = (ghost['pos'][0] + direction.value[0]) % self.width
            ghost['pos'][1] += direction.value[1]
            ghost['stuck_counter'] = 0
            return True
        else:
            ghost['stuck_counter'] += 1
            return False
    
    def generate_ghost_actions(self, random_values: List[float]) -> List[int]:
        """Generate ghost actions based on random values [0,1] for each ghost"""
        ghost_actions = []
        
        # Only process the first ghost (red ghost)
        if len(random_values) > 0 and len(self.ghosts) > 0:
            rand_val = random_values[0]
            possible_actions = self.get_ghost_possible_actions(0)
            ghost = self.ghosts[0]
            
            # Simple AI logic based on random value
            if ghost['stuck_counter'] > 3:
                # If stuck, try to get unstuck with higher probability of direction change
                if rand_val < 0.8:  # 80% chance to change direction when stuck
                    valid_moves = [a for a in possible_actions if a != 0]
                    if valid_moves:
                        action_idx = int(rand_val * len(valid_moves))
                        action = valid_moves[min(action_idx, len(valid_moves) - 1)]
                    else:
                        action = 0  # Stay if no valid moves
                else:
                    action = 0  # Stay
            else:
                # Normal behavior
                if rand_val < 0.1:  # 10% chance to change direction
                    valid_moves = [a for a in possible_actions if a != 0]
                    if valid_moves:
                        action_idx = int((rand_val * 10) * len(valid_moves))  # Scale up since we're in 0.1 range
                        action = valid_moves[min(action_idx, len(valid_moves) - 1)]
                    else:
                        action = 0
                elif rand_val < 0.2:  # 10% chance to stay
                    action = 0
                else:
                    # Continue in current direction if possible
                    current_dir_action = self.direction_to_action(ghost['dir'])
                    if current_dir_action in possible_actions:
                        action = current_dir_action
                    else:
                        # Pick a random valid action
                        valid_moves = [a for a in possible_actions if a != 0]
                        if valid_moves:
                            action_idx = int(rand_val * len(valid_moves))
                            action = valid_moves[min(action_idx, len(valid_moves) - 1)]
                        else:
                            action = 0
            
            ghost_actions.append(action)
        
        return ghost_actions
    
    def direction_to_action(self, direction: Direction) -> int:
        """Convert Direction enum to action number"""
        direction_map = {
            Direction.NONE: 0,
            Direction.UP: 1,
            Direction.DOWN: 2,
            Direction.LEFT: 3,
            Direction.RIGHT: 4
        }
        return direction_map.get(direction, 0)
    
    def move_ghosts_with_actions(self, ghost_actions: List[int]):
        """Move ghosts using provided actions"""
        if self.game_over:
            return
        
        # Only process the first ghost action
        if len(ghost_actions) > 0 and len(self.ghosts) > 0:
            self.apply_ghost_action(0, ghost_actions[0])
        
        # Check collisions with pacman
        for ghost in self.ghosts:
            if ghost['pos'] == self.pacman_pos:
                if self.power_mode > 0:
                    # Ghost eaten - respawn in center
                    ghost['pos'] = [self.width // 2, self.height // 2]
                    self.score += 200
                else:
                    self.game_over = True
        
        # Decrease power mode
        if self.power_mode > 0:
            self.power_mode -= 1
    
    def move_ghosts(self):
        """Original random ghost movement for backward compatibility"""
        if self.game_over:
            return
        
        # Generate random value for the single ghost
        random_values = [random.random()]
        ghost_actions = self.generate_ghost_actions(random_values)
        self.move_ghosts_with_actions(ghost_actions)
    
    def get_frame(self, pixels_per_cell=4) -> np.ndarray:
        """Generate RGB frame of current game state with specified pixels per cell"""
        frame_height = self.height * pixels_per_cell
        frame_width = self.width * pixels_per_cell
        frame = np.zeros((frame_height, frame_width, 3), dtype=np.uint8)
        
        # Draw maze
        for y in range(self.height):
            for x in range(self.width):
                pixel_x = x * pixels_per_cell
                pixel_y = y * pixels_per_cell
                
                if self.maze[y, x] == 1:  # Wall
                    frame[pixel_y:pixel_y+pixels_per_cell, pixel_x:pixel_x+pixels_per_cell] = [0, 0, 255]  # Blue
                elif self.maze[y, x] == 2:  # Pellet - changed to white
                    # Center the pellet dot
                    center_offset = max(1, pixels_per_cell // 4)
                    pellet_size = max(1, pixels_per_cell // 2)
                    start_x = pixel_x + center_offset
                    end_x = pixel_x + center_offset + pellet_size
                    start_y = pixel_y + center_offset
                    end_y = pixel_y + center_offset + pellet_size
                    frame[start_y:end_y, start_x:end_x] = [255, 255, 255]  # White dot
                elif self.maze[y, x] == 3:  # Power pellet - changed to white
                    # Larger pellet
                    center_offset = max(1, pixels_per_cell // 6)
                    pellet_size = max(2, (pixels_per_cell * 2) // 3)
                    start_x = pixel_x + center_offset
                    end_x = pixel_x + center_offset + pellet_size
                    start_y = pixel_y + center_offset
                    end_y = pixel_y + center_offset + pellet_size
                    frame[start_y:end_y, start_x:end_x] = [255, 255, 255]  # Bigger white dot
        
        # Draw ghost (only red ghost)
        ghost_colors = [
            [255, 0, 0],    # Red
        ]
        
        for i, ghost in enumerate(self.ghosts):
            x, y = ghost['pos']
            pixel_x = x * pixels_per_cell
            pixel_y = y * pixels_per_cell
            color = ghost_colors[0]  # Always red since we only have one ghost
            
            if self.power_mode > 0:
                color = [0, 0, 150]  # Dark blue when vulnerable
            
            # Ghost body (leave 1 pixel border if possible)
            border = max(0, pixels_per_cell // 8)
            frame[pixel_y+border:pixel_y+pixels_per_cell-border, 
                  pixel_x+border:pixel_x+pixels_per_cell-border] = color
        
        # Draw Pacman
        x, y = self.pacman_pos
        pixel_x = x * pixels_per_cell
        pixel_y = y * pixels_per_cell
        
        # Pacman body (leave 1 pixel border if possible)
        border = max(0, pixels_per_cell // 8)
        frame[pixel_y+border:pixel_y+pixels_per_cell-border, 
              pixel_x+border:pixel_x+pixels_per_cell-border] = [255, 255, 0]  # Yellow
        
        return frame

class PacManDataCollector:
    def __init__(self, width=21, height=21, frame_size=(96, 96)):
        pygame.init()
        self.game = GameState(width, height)
        self.frame_size = frame_size
        
        # Calculate pixels per cell to achieve target frame size
        # We'll use the larger dimension and ensure it's evenly divisible
        self.pixels_per_cell = max(frame_size[0] // width, frame_size[1] // height)
        
        # Display window (larger for better visibility during data collection)
        self.screen_width = width * 16 * 2  # 2x scale for display
        self.screen_height = height * 16 * 2
        self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))
        pygame.display.set_caption(f"Pac-Man Data Collector - {frame_size[0]}x{frame_size[1]} Resolution")
        self.clock = pygame.time.Clock()
        
        self.transitions = []
        self.current_direction = Direction.NONE
        
        print(f"Using pixels_per_cell={self.pixels_per_cell} to achieve ~{frame_size[0]}x{frame_size[1]} resolution")
        actual_width = width * self.pixels_per_cell
        actual_height = height * self.pixels_per_cell
        print(f"Actual frame size will be: {actual_width}x{actual_height}")
        
    def get_user_action(self, keys) -> Direction:
        if keys[pygame.K_UP]:
            return Direction.UP
        elif keys[pygame.K_DOWN]:
            return Direction.DOWN
        elif keys[pygame.K_LEFT]:
            return Direction.LEFT
        elif keys[pygame.K_RIGHT]:
            return Direction.RIGHT
        return Direction.NONE
    
    def get_user_action_number(self, action):
        if action == Direction.UP:
            return 1
        elif action == Direction.DOWN:
            return 2
        elif action == Direction.LEFT:
            return 3
        elif action == Direction.RIGHT:
            return 4
        return 0  # NOOP
    
    def get_frame_at_target_resolution(self) -> np.ndarray:
        """Get frame rendered at target resolution"""
        return self.game.get_frame(pixels_per_cell=self.pixels_per_cell)
    
    def collect_data(self, max_frames: int = 10000) -> None:
        frame_count = 0
        running = True
        
        print(f"Starting data collection with {self.frame_size[0]}x{self.frame_size[1]} resolution frames.")
        print("Use arrow keys to control Pac-Man.")
        print("Press R to reset the game.")
        print("Close window or press ESC to stop.")
        
        try:
            while running and frame_count < max_frames:
                reset_pressed = False
                
                # Handle events
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        running = False
                    elif event.type == pygame.KEYDOWN:
                        if event.key == pygame.K_ESCAPE:
                            running = False
                        elif event.key == pygame.K_r:
                            reset_pressed = True
                            self.game.reset()
                
                # Get current frame at target resolution
                current_frame = self.get_frame_at_target_resolution()
                
                # Get user input
                keys = pygame.key.get_pressed()
                user_action = self.get_user_action(keys)
                
                # Generate ghost actions based on game state (only one ghost now)
                if self.game.game_over:
                    # When game is over, set ghost action to 0 (stay)
                    ghost_actions = [0]
                    ghost_random_values = [0.0]
                else:
                    # Generate random value for the single ghost
                    ghost_random_values = [random.random()]
                    # Generate ghost action based on random value
                    ghost_actions = self.game.generate_ghost_actions(ghost_random_values)
                
                # Store previous state
                prev_score = self.game.score
                prev_game_over = self.game.game_over
                
                # Update game with recorded actions (only if not reset)
                if not reset_pressed:
                    self.game.move_pacman(user_action)
                    self.game.move_ghosts_with_actions(ghost_actions)
                
                # Get next frame at target resolution
                next_frame = self.get_frame_at_target_resolution()
                
                # Calculate reward
                reward = self.game.score - prev_score
                if self.game.game_over and not prev_game_over:
                    reward = -100 if not self.game.won else 1000
                
                # Store transition with all action information including reset
                if frame_count > 0:  # Skip first frame
                    self.transitions.append({
                        'frame': current_frame,
                        'user_action': self.get_user_action_number(user_action),
                        'ghost_actions': ghost_actions.copy(),  # [action1] - only one ghost
                        'ghost_random_values': ghost_random_values.copy(),  # The random value that generated the action
                        'reset': 1 if reset_pressed else 0,  # Track reset input
                        'next_frame': next_frame,
                        'reward': reward,
                        'score': self.game.score,
                        'game_over': self.game.game_over
                    })
                
                # Display frame (use native resolution for display, not target resolution)
                display_frame = self.game.get_frame(pixels_per_cell=16) # Use a fixed larger size for display
                display_surface = pygame.surfarray.make_surface(np.transpose(display_frame, (1, 0, 2)))
                display_surface = pygame.transform.scale(display_surface, (self.screen_width, self.screen_height))
                self.screen.blit(display_surface, (0, 0))
                
                # Draw UI
                font = pygame.font.Font(None, 36)
                score_text = font.render(f"Score: {self.game.score}", True, (255, 255, 255))
                frames_text = font.render(f"Frames: {frame_count}/{max_frames}", True, (255, 255, 255))
                resolution_text = font.render(f"Data Resolution: {self.frame_size[0]}x{self.frame_size[1]}", True, (255, 255, 255))
                
                # Show current actions and reset status
                if frame_count > 0 and len(self.transitions) > 0:
                    last_transition = self.transitions[-1]
                    actions_text = font.render(f"User: {last_transition['user_action']}, Ghost: {last_transition['ghost_actions']}", True, (255, 255, 255))
                    reset_text = font.render(f"Reset: {last_transition['reset']}", True, (255, 255, 0) if last_transition['reset'] else (255, 255, 255))
                    actual_size_text = font.render(f"Actual frame size: {current_frame.shape}", True, (255, 255, 255))
                    self.screen.blit(actions_text, (10, 170))
                    self.screen.blit(reset_text, (10, 210))
                    self.screen.blit(actual_size_text, (10, 250))
                
                self.screen.blit(score_text, (10, 10))
                self.screen.blit(frames_text, (10, 50))
                self.screen.blit(resolution_text, (10, 90))
                
                if self.game.game_over:
                    game_over_text = font.render("GAME OVER - Press R to restart", True, (255, 0, 0))
                    self.screen.blit(game_over_text, (10, 130))
                
                pygame.display.flip()
                self.clock.tick(10)  # 10 FPS for human playability
                
                frame_count += 1
                
                if frame_count % 1000 == 0 and frame_count > 0:
                    print(f"Collected {frame_count}/{max_frames} frames at {self.frame_size[0]}x{self.frame_size[1]} resolution")
                    if len(self.transitions) > 0:
                        print(f"Sample ghost action: {self.transitions[-1]['ghost_actions']}")
                        print(f"Reset count: {sum(1 for t in self.transitions if t['reset'] == 1)}")
                        print(f"Frame shape: {self.transitions[-1]['frame'].shape}")
        
        finally:
            pygame.quit()
            
        self.save_data(frame_count)
    
    def save_data(self, frame_count: int):
        if len(self.transitions) == 0:
            print("No data collected!")
            return
        
        print(f"Saving {len(self.transitions)} transitions at {self.frame_size[0]}x{self.frame_size[1]} resolution...")
        
        # Convert to numpy arrays
        frames = np.array([t['frame'] for t in self.transitions], dtype=np.uint8)
        user_actions = np.array([t['user_action'] for t in self.transitions], dtype=np.int8)
        ghost_actions = np.array([t['ghost_actions'] for t in self.transitions], dtype=np.int8)
        ghost_random_values = np.array([t['ghost_random_values'] for t in self.transitions], dtype=np.float32)
        resets = np.array([t['reset'] for t in self.transitions], dtype=np.int8)
        next_frames = np.array([t['next_frame'] for t in self.transitions], dtype=np.uint8)
        rewards = np.array([t['reward'] for t in self.transitions], dtype=np.float32)
        scores = np.array([t['score'] for t in self.transitions], dtype=np.int32)
        game_overs = np.array([t['game_over'] for t in self.transitions], dtype=bool)
        
        print(f"Frame shape: {frames.shape}")
        print(f"User actions shape: {user_actions.shape}")
        print(f"Ghost actions shape: {ghost_actions.shape}")
        print(f"Ghost random values shape: {ghost_random_values.shape}")
        print(f"Resets shape: {resets.shape}")
        print(f"Total resets recorded: {np.sum(resets)}")
        
        # Save to HDF5
        filename = f'pacman_single_ghost_{self.frame_size[0]}x{self.frame_size[1]}_{len(self.transitions)}_frames_with_reset_1.h5'
        with h5py.File(filename, 'w') as f:
            f.create_dataset('frames', data=frames, compression='gzip')
            f.create_dataset('user_actions', data=user_actions, compression='gzip')
            f.create_dataset('ghost_actions', data=ghost_actions, compression='gzip')
            f.create_dataset('ghost_random_values', data=ghost_random_values, compression='gzip')
            f.create_dataset('resets', data=resets, compression='gzip')
            f.create_dataset('next_frames', data=next_frames, compression='gzip')
            f.create_dataset('rewards', data=rewards, compression='gzip')
            f.create_dataset('scores', data=scores, compression='gzip')
            f.create_dataset('game_overs', data=game_overs, compression='gzip')
            
            # Metadata
            f.attrs['total_frames'] = len(self.transitions)
            f.attrs['frame_shape'] = frames.shape[1:]
            f.attrs['frame_resolution'] = f"{self.frame_size[0]}x{self.frame_size[1]}"
            f.attrs['collection_date'] = time.strftime('%Y-%m-%d %H:%M:%S')
            f.attrs['game_type'] = 'pacman_single_red_ghost_white_pellets'
            f.attrs['fps'] = 10
            f.attrs['num_ghosts'] = len(self.game.ghosts)
            f.attrs['total_resets'] = int(np.sum(resets))
            f.attrs['action_encoding'] = 'user: 0=none,1=up,2=down,3=left,4=right; ghost: 0=stay,1=up,2=down,3=left,4=right; reset: 0=no_reset,1=reset'
        
        print(f"Successfully saved data to {filename}")
        print(f"Dataset includes frames at {self.frame_size[0]}x{self.frame_size[1]} resolution:")
        print("- frames: Current game frames")
        print("- user_actions: Player actions (0-4)")
        print("- ghost_actions: Single ghost action [g1] (0-4)")
        print("- ghost_random_values: Random value [0,1] that generated ghost action")
        print("- resets: Reset input (0=normal play, 1=reset pressed)")
        print("- next_frames: Resulting frames after actions")
        print("- Ghost action is set to [0] when game is over")
        print("- Pellets are now white instead of yellow")
        print(f"- Total resets captured: {np.sum(resets)}")

def main():
    # Create collector with 40x40 resolution frames on a 10x10 grid
    collector = PacManDataCollector(width=10, height=10, frame_size=(40, 40))
    collector.collect_data(max_frames=40000)

if __name__ == "__main__":
    main()